# Phase 4 — Expected Points Models

## 1. Load OOF predictions

In [ ]:
import json
from pathlib import Path
import pandas as pd
from fpl_predictor.config import HISTORICAL_ML_DIR

oof = pd.read_csv(HISTORICAL_ML_DIR / 'oof_predictions.csv')
results = pd.read_csv(HISTORICAL_ML_DIR / 'model_results.csv')
positions = pd.read_csv(HISTORICAL_ML_DIR / 'model_results_by_position.csv')
gameweeks = pd.read_csv(HISTORICAL_ML_DIR / 'model_results_by_gw.csv')
calibration = pd.read_csv(HISTORICAL_ML_DIR / 'calibration_results.csv')
residuals = pd.read_csv(HISTORICAL_ML_DIR / 'residual_analysis.csv')
summary = json.loads((HISTORICAL_ML_DIR / 'phase4_summary.json').read_text())

## 2. Model comparison

In [ ]:
display(results.sort_values(['split', 'mae'])[['split', 'model', 'mode', 'feature_set', 'mae', 'rmse', 'spearman']])

## 3. Performance by position

In [ ]:
display(positions.pivot_table(index=['split', 'model'], columns='position', values='mae').round(3))

## 4. MAE/RMSE/Spearman

In [ ]:
display(results[['split', 'model', 'mae', 'rmse', 'spearman']].dropna().sort_values(['split', 'mae']))

## 5. Top-K and NDCG

In [ ]:
display(results[['split', 'model', 'top_10_precision', 'top_25_precision', 'top_50_precision', 'ndcg_10', 'ndcg_25', 'ndcg_50']].dropna(subset=['ndcg_25']))

## 6. High-ceiling performance

In [ ]:
display(results.query("split == 'test'")[["model", "ceiling_8_precision", "ceiling_8_recall", "ceiling_10_precision", "ceiling_10_recall", "ceiling_15_recall"]])

## 7. Calibration

In [ ]:
display(calibration.query("split == 'test'"))

## 8. Residual analysis

In [ ]:
display(residuals.query("split == 'test' and dimension in ['price_band', 'minutes_history_band']").sort_values(['dimension', 'mae']))

## 9. Early-season performance

In [ ]:
display(residuals.query("split == 'test' and dimension == 'season_stage'"))

## 10. Position-specific models

In [ ]:
display(results.query("split == 'validation' and mode == 'position_specific'"))

## 11. Uncertainty bands

In [ ]:
display(pd.DataFrame(summary['uncertainty']['bands']))
test_predictions = pd.read_csv(HISTORICAL_ML_DIR / 'test_predictions_with_uncertainty.csv')
display(test_predictions.nlargest(15, 'predicted_points')[['player_name', 'position', 'predicted_points', 'lower_bound', 'upper_bound']])

## 12. Final model selection

The frozen production configuration was selected only from 2024/25 using the documented multi-metric score. Treat the empirical intervals as descriptive ranges, and inspect high-ceiling recall before using xPts for future captain or transfer decisions.

In [ ]:
display(summary['selected_model'])
display(pd.DataFrame([summary['test_metrics']]))